# v1 — 청각 마스킹 기반 섭동 배치 (Enkidu 개조)

**질문:** Enkidu 섭동을 원본 음성의 심리음향 마스킹 임계값 θ(f,t)에 맞춰 배치하면, 사용성(명료도)을 올리면서 보호를 유지할 수 있는가?

**바꾼 것:** 패치를 스펙트로그램에 더할 때 주파수 빈·프레임마다 상한 `κ·√θ(f,t)`를 두고, 넘는 부분만 잘라낸다(위상 유지). κ=∞이면 원본 Enkidu와 같다.

**그대로 둔 것:** 패치 모양(513×30), 타일링, `noise_level=0.4`, `mask_ratio=0.3`, Adam(lr 0.1, weight_decay 0.2), 50 epochs, 대리 모델 ECAPA, 학습 70% / 평가 30% 분할.

| 단계 | 내용 | 결과 위치 (`xtts_exp/`) |
|---|---|---|
| v1-a | 사후 마스킹 κ=1 | `mask_exp/k1/` |
| v1-b | 사후 마스킹 κ 스윕 vs 균일 축소 (입력 단계) | `mask_exp/sweep.csv` |
| v1-c | 균일 축소 + Resemblyzer | `mask_exp/uniform_resem.csv` |
| 진단 | Wiener 필터 분해 | `mask_exp/decompose_wiener.csv` |
| v1-d | **학습 단계 마스킹** κ=16, κ=16(지각 손실 끔), κ=64 | `mask_exp/train_k*/` |
| 정화·복제 | κ=64 보호본 → `02`, `03` 노트북 | `mask_exp/train_k64/D3`, `runs/enkidu_mask_k64/` |

결과는 `results/README.md`에 정리돼 있다. 기존 D1~D4, `runs/run1/enkidu_ufp`의 원본 패치는 **읽기만** 한다.


## 1. 환경 · 데이터 · 원본 패치 (`01_protect_enkidu`와 같음)

In [ ]:
import torch, sys
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "런타임 유형을 GPU로 바꿔 주세요"

In [ ]:
# speechbrain 1.0 이상이어야 speechbrain.inference 경로가 있다
!pip -q install "speechbrain>=1.0.0" pystoi
import speechbrain
print("speechbrain :", speechbrain.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

RUN_ID = 'run1'        # 설정을 바꿔 다시 돌릴 때는 여기만 바꾼다

ROOT      = Path('/content/drive/MyDrive/xtts_exp')
SEG_DIR   = ROOT / 'D1'                           # 원본 (화자별 폴더)
PROT_DIR  = ROOT / 'D2'                           # 보호본 출력 (같은 구조)
RUN_DIR   = ROOT / 'runs' / RUN_ID                # 이 회차의 산출물
UFP_DIR   = RUN_DIR / 'enkidu_ufp'                # 학습한 패치 저장

for d in (PROT_DIR, UFP_DIR, RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert SEG_DIR.exists(), f"D1 폴더가 없습니다: {SEG_DIR}"
print("D1 (원본)  :", SEG_DIR)
print("D2 (보호본):", PROT_DIR)
print("runs       :", RUN_DIR)

In [ ]:
%cd /content
!rm -rf Enkidu
!git clone -q https://github.com/NoobCodeNameless/Enkidu.git
%cd /content/Enkidu
!ls

In [ ]:
import sys
sys.path.insert(0, '/content/Enkidu')

from core import Enkidu
from wavdataset import WaveformPrivacyDataset
print("import OK")

In [ ]:
import torchaudio
import soundfile as sf

SR = 16000

ds = WaveformPrivacyDataset(
    dataset_dir=str(SEG_DIR),
    sample_rate=SR,
    mono=True,
    wav_format='wav',     # 기본 예시는 LibriSpeech라 flac이다
)

print(f"화자 {len(ds.speakers)}명, 전체 조각 {len(ds)}개\n")
print(f"{'화자':<14}{'조각':>6}{'총 길이':>10}{'평균':>8}   판정")
print("-" * 54)

for spk in ds.speakers:
    paths = ds.speaker_samples[spk]
    total = 0.0
    for p in paths:
        total += sf.info(p).duration      # torchaudio.info는 새 버전에서 제거됨
    n = len(paths)
    avg = total / n if n else 0
    verdict = "충분" if n >= 40 else ("부족 - 추가 녹음 권장" if n < 20 else "빈약")
    print(f"{spk:<14}{n:>6}{total/60:>9.1f}분{avg:>7.1f}초   {verdict}")

In [ ]:
NOISE_LEVEL  = 0.4
FRAME_LENGTH = 30
MASK_RATIO   = 0.3
STEPS        = 50
ALPHA        = 0.1
TRAIN_RATIO  = 0.7
DEVICE       = 'cuda:0'

def split_speaker(spk):
    """화자의 조각을 학습용 / 평가용으로 겹치지 않게 나눈다."""
    paths = sorted(ds.speaker_samples[spk])
    k = max(1, int(len(paths) * TRAIN_RATIO))
    return paths[:k], paths[k:]

for spk in ds.speakers:
    tr, ev = split_speaker(spk)
    print(f"{spk:<14} 학습 {len(tr):>3}개   평가 {len(ev):>3}개")

In [ ]:
import io, time, contextlib
from speechbrain.inference import SpeakerRecognition
import torch.nn.functional as F

# ECAPA 하나를 대리 모델과 검증에 같이 쓴다.
# Enkidu가 공략하는 모델이므로 여기 수치는 낙관적으로 나온다는 점을 기억할 것.
ecapa = SpeakerRecognition.from_hparams(
    'speechbrain/spkrec-ecapa-voxceleb',
    savedir='/content/pretrained/ecapa',
    run_opts={"device": DEVICE},
)

def load_wav(path):
    w, sr = torchaudio.load(path)
    if w.shape[0] > 1:
        w = w.mean(dim=0, keepdim=True)
    if sr != SR:
        w = torchaudio.transforms.Resample(sr, SR)(w)
    return w

@torch.no_grad()
def ecapa_sim(a, b):
    ea = ecapa.encode_batch(a.reshape(1, -1).to(DEVICE)).reshape(1, -1)
    eb = ecapa.encode_batch(b.reshape(1, -1).to(DEVICE)).reshape(1, -1)
    return F.cosine_similarity(ea, eb).item()

아래 칸은 `runs/run1/enkidu_ufp/`의 기존 패치를 불러온다. `[화자] 기존 패치 사용`이 화자 수만큼 찍혀야 한다. **`RUN_ID`를 바꾸면 재학습하니 주의.**

In [ ]:
ufps = {}

for spk in ds.speakers:
    pt = UFP_DIR / f'{spk}.pt'
    if pt.exists():
        d = torch.load(pt)
        ufps[spk] = (d['real'], d['imag'])
        print(f"[{spk}] 기존 패치 사용 ({pt.name})")
        continue

    train_paths, _ = split_speaker(spk)
    samples = [load_wav(p) for p in train_paths]

    model = Enkidu(
        model=ecapa,
        steps=STEPS,
        alpha=ALPHA,
        mask_ratio=MASK_RATIO,
        frame_length=FRAME_LENGTH,
        noise_level=NOISE_LEVEL,
        noise_smooth=True,
        device=DEVICE,
    )

    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()):   # 샘플마다 찍는 로그를 막는다
        nr, ni = model(samples)
    dt = time.time() - t0

    torch.save({'real': nr, 'imag': ni,
                'noise_level': NOISE_LEVEL, 'frame_length': FRAME_LENGTH,
                'mask_ratio': MASK_RATIO, 'steps': STEPS,
                'n_train': len(samples)}, pt)
    ufps[spk] = (nr, ni)
    print(f"[{spk}] 학습 {len(samples)}개 · {dt/60:.1f}분 · 저장 {pt.name}")

## 2. 마스킹 모듈과 평가 도구

In [ ]:
%%writefile /content/drive/MyDrive/xtts_exp/psychomask.py
"""
Psychoacoustic masking threshold (MPEG-1 Psychoacoustic Model 1 style).
Reference formulation: Qin et al., "Imperceptible, Robust, and Targeted
Adversarial Examples for ASR", ICML 2019.

theta(f, t) = the power, per frequency bin per frame, that can be added
to x without being audible.  Depends ONLY on x, so it can be precomputed
once per utterance and treated as a constant inside the training loop.
"""

import numpy as np


# ---------------------------------------------------------------- helpers
def hz_to_bark(f):
    f = np.asarray(f, dtype=np.float64)
    return 13.0 * np.arctan(0.00076 * f) + 3.5 * np.arctan((f / 7500.0) ** 2)


def absolute_threshold_hearing(f):
    """ATH in dB SPL. f in Hz. Clamped below 20 Hz to avoid blow-up."""
    f = np.maximum(np.asarray(f, dtype=np.float64), 20.0)
    k = f / 1000.0
    return (3.64 * k ** (-0.8)
            - 6.5 * np.exp(-0.6 * (k - 3.3) ** 2)
            + 1e-3 * k ** 4)


def _spreading(dz, p_masker_db):
    """
    Two-slope spreading function, dB.
    dz = bark(maskee) - bark(masker).  Lower slope +27 dB/bark,
    upper slope level-dependent.
    """
    upper = -27.0 + 0.37 * np.maximum(p_masker_db - 40.0, 0.0)
    return np.where(dz < 0, 27.0 * dz, upper * dz)


# ---------------------------------------------------------- main routine
def masking_threshold(
    x,
    sr=16000,
    n_fft=1024,
    hop=256,
    win=None,
    max_maskers=48,
    center=True,
):
    """
    Parameters
    ----------
    x      : 1-D float array, waveform in [-1, 1]
    center : pad n_fft//2 on both sides, so the frame count matches
             torch.stft(..., center=True).  Keep True when the result is
             multiplied against a torch STFT.
    Returns
    -------
    theta : (n_bins, n_frames) float64, masking threshold in POWER units,
            on the same scale as |STFT(x)|**2 after the 96 dB normalisation
            is undone -- i.e. directly comparable to |STFT(x)|**2.
    """
    if win is None:
        win = np.hanning(n_fft)

    x = np.asarray(x, dtype=np.float64).reshape(-1)
    if center:
        x = np.pad(x, (n_fft // 2, n_fft // 2), mode="reflect")

    # ---- STFT ------------------------------------------------------
    n = len(x)
    n_frames = 1 + max(0, (n - n_fft) // hop)
    if n_frames < 1:
        x = np.pad(x, (0, n_fft - n))
        n_frames = 1
    idx = np.arange(n_fft)[None, :] + hop * np.arange(n_frames)[:, None]
    frames = x[idx] * win[None, :]
    X = np.fft.rfft(frames, n=n_fft, axis=1)          # (T, K)
    mag2 = np.abs(X) ** 2
    n_bins = X.shape[1]

    freqs = np.fft.rfftfreq(n_fft, 1.0 / sr)
    barks = hz_to_bark(freqs)
    ath = absolute_threshold_hearing(freqs)            # dB SPL

    # ---- PSD normalised so that the global max is 96 dB SPL --------
    eps = 1e-20
    psd_raw = 10.0 * np.log10(mag2 + eps)              # dB, arbitrary ref
    offset = 96.0 - psd_raw.max()
    psd = psd_raw + offset                             # dB SPL

    theta_db = np.empty((n_frames, n_bins), dtype=np.float64)

    for t in range(n_frames):
        p = psd[t]

        # ---- tonal maskers: local peaks standing >=7 dB above
        #      their neighbourhood -------------------------------------
        is_peak = np.zeros(n_bins, dtype=bool)
        is_peak[1:-1] = (p[1:-1] > p[:-2]) & (p[1:-1] >= p[2:])
        cand = np.where(is_peak)[0]

        keep = []
        for k in cand:
            if k < 2 or k > n_bins - 3:
                continue
            # neighbourhood width grows with frequency (MPEG dk rule)
            if k < 63:
                dks = (2,)
            elif k < 127:
                dks = (2, 3)
            else:
                dks = (2, 3, 4, 5, 6)
            nb = []
            for dk in dks:
                if 0 <= k - dk and k + dk < n_bins:
                    nb.extend((p[k - dk], p[k + dk]))
            if nb and p[k] >= max(nb) + 7.0:
                keep.append(k)

        if len(keep) == 0:
            theta_db[t] = ath
            continue

        # strongest maskers only -- keeps the O(K*M) loop bounded
        keep = np.array(keep)
        if len(keep) > max_maskers:
            keep = keep[np.argsort(p[keep])[-max_maskers:]]

        # tonal masker level = sum of the peak and its two neighbours
        p_mask = 10.0 * np.log10(
            np.sum(10.0 ** (p[np.stack([keep - 1, keep, keep + 1])] / 10.0),
                   axis=0) + eps
        )                                              # (M,)

        # drop maskers already below the hearing threshold
        alive = p_mask >= ath[keep]
        if not alive.any():
            theta_db[t] = ath
            continue
        keep, p_mask = keep[alive], p_mask[alive]

        # ---- individual masking thresholds ------------------------
        dz = barks[None, :] - barks[keep][:, None]     # (M, K)
        T = (p_mask[:, None]
             - 0.275 * barks[keep][:, None]
             + _spreading(dz, p_mask[:, None])
             - 6.025)

        # only within +-3 bark does a masker contribute
        T = np.where(np.abs(dz) <= 3.0, T, -np.inf)

        # ---- global threshold: power-sum of maskers and ATH --------
        with np.errstate(over="ignore"):
            tot = np.sum(10.0 ** (T / 10.0), axis=0) + 10.0 ** (ath / 10.0)
        theta_db[t] = 10.0 * np.log10(tot + eps)

    # ---- back to the power scale of |STFT(x)|**2 ------------------
    theta = 10.0 ** ((theta_db - offset) / 10.0)
    return theta.T                                     # (K, T)


# ------------------------------------------------------- apply the gain
def shape_perturbation(delta_tiled, theta, kappa=1.0, noise_level=1.0):
    """
    Clip the perturbation to the masking ceiling, per bin.

        delta_out = noise_level * delta * min(1, kappa*sqrt(theta) / |noise_level*delta|)

    The patch keeps its learned magnitude wherever that already sits below
    what the ear can hide; it is pulled down only where it would stick out.
    Phase is never touched.

    delta_tiled : (K, T) complex, the tiled UFP
    theta       : (K, T) real, from masking_threshold()
    noise_level : Enkidu's existing global scale -- keep it
    kappa       : the new knob.  kappa -> inf reproduces plain Enkidu.

    Returns (delta_out, clip_fraction)
    """
    d = noise_level * delta_tiled
    mag = np.abs(d)
    ceil = kappa * np.sqrt(theta)
    gain = np.minimum(1.0, ceil / np.maximum(mag, 1e-12))
    clipped = float((gain < 0.999).mean())
    return d * gain, clipped


# ------------------------------------------- numpy STFT / iSTFT (matched)
def _stft(x, n_fft=1024, hop=256):
    x = np.pad(np.asarray(x, np.float64).reshape(-1),
               (n_fft // 2, n_fft // 2), mode="reflect")
    win = np.hanning(n_fft)
    nf = 1 + (len(x) - n_fft) // hop
    idx = np.arange(n_fft)[None, :] + hop * np.arange(nf)[:, None]
    return np.fft.rfft(x[idx] * win[None, :], axis=1).T     # (K, T)


def _istft(S, length, n_fft=1024, hop=256):
    win = np.hanning(n_fft)
    frames = np.fft.irfft(S.T, n=n_fft, axis=1) * win[None, :]
    nf = frames.shape[0]
    total = n_fft + hop * (nf - 1)
    y = np.zeros(total); wsum = np.zeros(total)
    for t in range(nf):
        s = t * hop
        y[s:s + n_fft] += frames[t]
        wsum[s:s + n_fft] += win ** 2
    y /= np.maximum(wsum, 1e-8)
    y = y[n_fft // 2: n_fft // 2 + length]
    if len(y) < length:
        y = np.pad(y, (0, length - len(y)))
    return y


def apply_masked(x, y, kappa=1.0, sr=16000, n_fft=1024, hop=256):
    """
    x : clean waveform (1-D numpy)
    y : Enkidu output for the same x (1-D numpy), i.e. x + delta
    Returns (y_masked, info)

    Because STFT is linear, delta = y - x exactly.  delta is re-analysed,
    clipped bin-by-bin to kappa*sqrt(theta(x)), and added back.
    kappa = inf returns y unchanged.
    """
    x = np.asarray(x, np.float64).reshape(-1)
    y = np.asarray(y, np.float64).reshape(-1)
    n = min(len(x), len(y)); x, y = x[:n], y[:n]
    d = y - x
    if not np.isfinite(kappa):
        return y, {"clip_frac": 0.0, "energy_kept": 1.0}

    theta = masking_threshold(x, sr=sr, n_fft=n_fft, hop=hop, center=True)
    D = _stft(d, n_fft, hop)
    T = min(D.shape[1], theta.shape[1])
    D, theta = D[:, :T], theta[:, :T]
    mag = np.abs(D)
    gain = np.minimum(1.0, kappa * np.sqrt(theta) / np.maximum(mag, 1e-12))
    d2 = _istft(D * gain, n, n_fft, hop)
    info = {"clip_frac": float((gain < 0.999).mean()),
            "energy_kept": float((d2 ** 2).sum() / max((d ** 2).sum(), 1e-20))}
    return x + d2, info

In [ ]:
sys.path.insert(0, '/content/drive/MyDrive/xtts_exp')
import importlib, psychomask
importlib.reload(psychomask)
from psychomask import apply_masked
print("psychomask OK")

In [ ]:
!pip -q install webrtcvad-wheels resemblyzer pystoi
import numpy as np, pandas as pd
from pystoi import stoi
from resemblyzer import VoiceEncoder, preprocess_wav

resem = VoiceEncoder(device='cuda')
def resem_sim(a, b):
    ea = resem.embed_utterance(preprocess_wav(a, source_sr=SR))
    eb = resem.embed_utterance(preprocess_wav(b, source_sr=SR))
    return float(np.dot(ea, eb))          # 정규화된 벡터라 내적 = 코사인

applier = Enkidu(model=ecapa, steps=1, alpha=ALPHA, mask_ratio=MASK_RATIO,
                 frame_length=FRAME_LENGTH, noise_level=NOISE_LEVEL,
                 noise_smooth=True, device=DEVICE)
print("평가 도구 준비됨")

## v1-a. 사후 마스킹 κ=1

학습된 원본 패치를 그대로 쓰고, 붙인 뒤에 마스킹 천장으로 자른다. 결과: ECAPA 0.958, STOI 0.967 — 보호가 거의 사라짐.

In [ ]:
KAPPA = 1.0

OUT_ROOT = ROOT / 'mask_exp' / f'k{KAPPA:g}'
K1_D2 = OUT_ROOT / 'D2'
K1_D2.mkdir(parents=True, exist_ok=True)
assert K1_D2.resolve() != (ROOT / 'D2').resolve(), "기존 D2 경로입니다"

rows = []
for spk in ds.speakers:
    nr, ni = ufps[spk]
    _, eval_paths = split_speaker(spk)
    (K1_D2 / spk).mkdir(parents=True, exist_ok=True)
    for p in eval_paths:
        x = load_wav(p)
        y = applier.add_noise(x, nr, ni, mask_ratio=0.0, random_offset=False, noise_smooth=True)
        ym, info = apply_masked(x[0].numpy(), y[0].detach().cpu().numpy(), kappa=KAPPA, sr=SR)
        torchaudio.save(str(K1_D2 / spk / Path(p).name), torch.from_numpy(ym).float().unsqueeze(0), SR)
        xn = x[0].numpy(); n = min(len(xn), len(ym))
        rows.append({'speaker': spk, 'sim': ecapa_sim(x, torch.from_numpy(ym).float()),
                     'stoi': stoi(xn[:n], ym[:n], SR, extended=False),
                     'energy_kept': info['energy_kept']})

k1 = pd.DataFrame(rows)
print(k1.groupby('speaker')[['sim', 'stoi']].mean().round(3).to_string())
print(f"전체  ECAPA {k1.sim.mean():.3f}  STOI {k1.stoi.mean():.3f}  섭동 에너지 유지 {k1.energy_kept.mean()*100:.1f}%")
k1.to_csv(OUT_ROOT / 'protection_check.csv', index=False)

## v1-b. 사후 마스킹 κ 스윕 vs 균일 축소 (대조군)

같은 Enkidu 출력에 (1) κ별 마스킹, (2) 섭동 전체를 c배로 균일 축소를 적용해 입력 단계 ECAPA 유사도와 STOI를 비교한다. **같은 STOI에서 마스킹 쪽 유사도가 더 낮아야** 마스킹이 효과 있는 것이다.

In [ ]:
KAPPAS  = [2, 4, 8, 16, 32, 64, np.inf]
UNIFORM = [0.25, 0.5, 0.75]

rows = []
for spk in ds.speakers:
    nr, ni = ufps[spk]
    _, eval_paths = split_speaker(spk)
    for p in eval_paths:
        x = load_wav(p)
        y = applier.add_noise(x, nr, ni, mask_ratio=0.0, random_offset=False, noise_smooth=True)
        xn = x[0].numpy().astype(np.float64)
        yn = y[0].detach().cpu().numpy().astype(np.float64)
        n  = min(len(xn), len(yn)); xn, yn = xn[:n], yn[:n]

        cands = []
        for k in KAPPAS:
            ym, info = apply_masked(xn, yn, kappa=k, sr=SR)
            cands.append(('mask', k, ym, info['energy_kept']))
        for c in UNIFORM:
            cands.append(('uniform', c, xn + c * (yn - xn), c ** 2))

        xt = torch.from_numpy(xn).float()
        for kind, val, ym, ek in cands:
            rows.append({'kind': kind, 'param': val, 'speaker': spk,
                         'sim': ecapa_sim(xt, torch.from_numpy(ym).float()),
                         'stoi': stoi(xn, ym, SR, extended=False), 'energy_kept': ek})
    print(f"[{spk}] 완료")

sweep = pd.DataFrame(rows)
print(sweep.groupby(['kind', 'param']).agg(sim=('sim', 'mean'), stoi=('stoi', 'mean'),
                                           energy_kept=('energy_kept', 'mean')).round(3).to_string())
sweep.to_csv(ROOT / 'mask_exp' / 'sweep.csv', index=False)

## v1-c. 균일 축소 + Resemblyzer, 들어보기

In [ ]:
from IPython.display import Audio, display

SCALES = [1.0, 0.75, 0.5, 0.25]           # 1.0 = 원본 Enkidu
rows, samples = [], []
for spk in ds.speakers:
    nr, ni = ufps[spk]
    _, eval_paths = split_speaker(spk)
    for i, p in enumerate(eval_paths):
        x = load_wav(p)
        y = applier.add_noise(x, nr, ni, mask_ratio=0.0, random_offset=False, noise_smooth=True)
        xn = x[0].numpy().astype(np.float32)
        yn = y[0].detach().cpu().numpy().astype(np.float32)
        n  = min(len(xn), len(yn)); xn, yn = xn[:n], yn[:n]
        for c in SCALES:
            ym = xn + c * (yn - xn)
            rows.append({'c': c, 'speaker': spk,
                         'ecapa': ecapa_sim(torch.from_numpy(xn), torch.from_numpy(ym)),
                         'resem': resem_sim(xn, ym),
                         'stoi':  stoi(xn, ym, SR, extended=False)})
        if i == 0:
            samples.append((spk, Path(p).name, xn, yn))
    print(f"[{spk}] 완료")

r = pd.DataFrame(rows)
print(r.groupby('c')[['ecapa', 'resem', 'stoi']].mean().round(3).sort_index(ascending=False).to_string())
r.to_csv(ROOT / 'mask_exp' / 'uniform_resem.csv', index=False)

for spk, name, xn, yn in samples:
    print(f"\n{spk} / {name}")
    print("원본");              display(Audio(xn, rate=SR))
    print("Enkidu 원본 (c=1)"); display(Audio(yn, rate=SR))
    print("c=0.5");             display(Audio(xn + 0.5 * (yn - xn), rate=SR))

## 진단. 보호는 패치에서 오는가, Wiener 필터에서 오는가

`add_noise(noise_smooth=True)`는 음성+섭동 전체에 Wiener 필터를 건다. A(패치+Wiener) / B(패치만) / C(Wiener만)를 비교한다. 결과: B ≈ A, C ≈ 원본 → 보호와 사용성 저하 모두 패치에서 온다.

In [ ]:
rows = []
for spk in ds.speakers:
    nr, ni = ufps[spk]
    zr, zi = torch.zeros_like(nr), torch.zeros_like(ni)
    _, eval_paths = split_speaker(spk)
    for p in eval_paths:
        x = load_wav(p)
        conds = {
            'A 패치+Wiener (원본 Enkidu)': applier.add_noise(x, nr, ni, mask_ratio=0.0,
                                            random_offset=False, noise_smooth=True),
            'B 패치만 (Wiener 끔)':       applier.add_noise(x, nr, ni, mask_ratio=0.0,
                                            random_offset=False, noise_smooth=False),
            'C Wiener만 (패치 없음)':     applier.add_noise(x, zr, zi, mask_ratio=0.0,
                                            random_offset=False, noise_smooth=True),
        }
        xn = x[0].numpy().astype(np.float32)
        for name, y in conds.items():
            yn = y[0].detach().cpu().numpy().astype(np.float32)
            n = min(len(xn), len(yn)); a, b = xn[:n], yn[:n]
            rows.append({'cond': name,
                         'ecapa': ecapa_sim(torch.from_numpy(a), torch.from_numpy(b)),
                         'resem': resem_sim(a, b),
                         'stoi':  stoi(a, b, SR, extended=False)})
    print(f"[{spk}] 완료")

dec = pd.DataFrame(rows)
print(dec.groupby('cond')[['ecapa', 'resem', 'stoi']].mean().round(3).to_string())
dec.to_csv(ROOT / 'mask_exp' / 'decompose_wiener.csv', index=False)

## v1-d. 학습 단계 마스킹 (MaskedEnkidu)

Enkidu를 상속해 `add_noise`만 바꾼다. 학습할 때도 매번 마스킹 천장으로 잘라서, 패치가 잘릴 것을 알고 학습되게 한다. Wiener는 끈다(진단에서 영향 없음 확인). `core/` 코드는 수정하지 않는다.

In [ ]:
import random, io, time, contextlib
from psychomask import masking_threshold
from core.enkidu import WienerFilter

class MaskedEnkidu(Enkidu):
    """학습과 적용 모두에서 섭동을 빈별 마스킹 천장 kappa*sqrt(theta)로 자른다."""
    def __init__(self, *args, kappa=16.0, **kw):
        super().__init__(*args, **kw)
        self.kappa = kappa
        self._cache = {}

    def _ceiling(self, x, T):
        w = x.detach().flatten().cpu().numpy().astype(np.float64)
        key = (w.size, round(float(w.sum()), 4), round(float(np.abs(w).sum()), 4))
        if key not in self._cache:
            th = masking_threshold(w, sr=self.sample_rate, n_fft=self.n_fft,
                                   hop=self.hop_length, center=True)
            self._cache[key] = torch.from_numpy(np.sqrt(th)).float()
        c = self._cache[key]
        if c.shape[1] < T:
            c = torch.cat([c, c[:, -1:].expand(-1, T - c.shape[1])], dim=1)
        return c[:, :T].unsqueeze(0).to(self.device)

    def add_noise(self, source_waveform, noise_real, noise_imag,
                  mask_ratio=0.0, random_offset=False, noise_smooth=False):
        src_dev = source_waveform.device
        x  = source_waveform.flatten().to(self.device)
        nr = noise_real.to(self.device)
        ni = noise_imag.to(self.device)

        S = torch.stft(x, n_fft=self.n_fft, hop_length=self.hop_length,
                       win_length=self.win_length, window=self.window,
                       return_complex=True).unsqueeze(0)
        T, L = S.shape[-1], self.frame_length

        Dr = torch.zeros_like(S.real)
        Di = torch.zeros_like(S.imag)
        if L > T:
            Dr[:, :, :T] = nr[:, :, :T] * self.noise_level
            Di[:, :, :T] = ni[:, :, :T] * self.noise_level
        else:
            n_patch = T // L
            offset = random.randint(0, T % L) if random_offset else 0
            keep = torch.rand((n_patch,)) >= mask_ratio
            if mask_ratio > 0 and keep.sum() == 0:
                keep[random.randint(0, n_patch - 1)] = True
            for i in range(n_patch):
                if keep[i]:
                    lo, hi = offset + i * L, offset + (i + 1) * L
                    Dr[:, :, lo:hi] = nr * self.noise_level
                    Di[:, :, lo:hi] = ni * self.noise_level

        mag  = torch.sqrt(Dr ** 2 + Di ** 2 + 1e-12)
        gain = torch.clamp(self.kappa * self._ceiling(x, T) / mag, max=1.0)
        S2 = torch.complex(S.real + Dr * gain, S.imag + Di * gain)

        if noise_smooth:
            S2 = WienerFilter()(S2)

        y = torch.istft(S2, n_fft=self.n_fft, hop_length=self.hop_length,
                        win_length=self.win_length, window=self.window,
                        length=x.size(-1)).reshape(1, -1)
        return y.clamp(-1.0, 1.0).to(src_dev)


class MaskedEnkiduNoPer(MaskedEnkidu):
    """지각 손실(λ)을 끈 버전"""
    def total_loss(self, clean_waveform, noisy_waveform, lambda_perceptual=0.1):
        return Enkidu.total_loss(self, clean_waveform, noisy_waveform, lambda_perceptual=0.0)

print("MaskedEnkidu 준비됨")

세 설정을 학습·평가한다. 화자마다 패치가 저장되므로 끊겨도 다시 돌리면 이어서 한다. 화자당 약 3분.

마지막 표의 `균일축소` 열은 같은 STOI에서 균일 축소(v1-c)의 값을 선형 보간한 것이다(STOI 0.80~0.89 구간에서만 유효). `ECAPA`가 그보다 **낮아야** 마스킹이 이긴 것이다.

In [ ]:
CONFIGS = [            # (kappa, 지각손실 사용)
    (16.0, True),
    (16.0, False),
    (64.0, True),         # 정화·복제로 보낸 설정
]

summary = []
for kappa, use_per in CONFIGS:
    tag  = f'train_k{kappa:g}' + ('' if use_per else '_noper')
    root = ROOT / 'mask_exp' / tag
    (root / 'ufp').mkdir(parents=True, exist_ok=True)
    cls  = MaskedEnkidu if use_per else MaskedEnkiduNoPer
    kw   = dict(model=ecapa, alpha=ALPHA, mask_ratio=MASK_RATIO,
                frame_length=FRAME_LENGTH, noise_level=NOISE_LEVEL,
                noise_smooth=False, device=DEVICE, kappa=kappa)

    pats = {}
    for spk in ds.speakers:
        pt = root / 'ufp' / f'{spk}.pt'
        if pt.exists():
            d = torch.load(pt); pats[spk] = (d['real'], d['imag']); continue
        tr, _ = split_speaker(spk)
        m = cls(steps=STEPS, **kw)
        t0 = time.time()
        with contextlib.redirect_stdout(io.StringIO()):
            nr, ni = m([load_wav(p) for p in tr])
        torch.save({'real': nr, 'imag': ni, 'kappa': kappa, 'perceptual': use_per}, pt)
        pats[spk] = (nr, ni)
        print(f"  [{tag}] {spk} 학습 {len(tr)}개 · {(time.time()-t0)/60:.1f}분")

    app, rows = cls(steps=1, **kw), []
    for spk in ds.speakers:
        nr, ni = pats[spk]
        _, ev = split_speaker(spk)
        (root / 'D2' / spk).mkdir(parents=True, exist_ok=True)
        for p in ev:
            x = load_wav(p)
            y = app.add_noise(x, nr, ni, mask_ratio=0.0, random_offset=False)
            torchaudio.save(str(root / 'D2' / spk / Path(p).name), y.cpu(), SR)
            a = x[0].numpy().astype(np.float32)
            b = y[0].detach().cpu().numpy().astype(np.float32)
            n = min(len(a), len(b)); a, b = a[:n], b[:n]
            rows.append({'ecapa': ecapa_sim(torch.from_numpy(a), torch.from_numpy(b)),
                         'resem': resem_sim(a, b),
                         'stoi':  stoi(a, b, SR, extended=False)})
    pd.DataFrame(rows).to_csv(root / 'eval.csv', index=False)
    r = pd.DataFrame(rows).mean(); s = r['stoi']
    summary.append({'설정': tag, 'ECAPA': r['ecapa'], 'Resem': r['resem'], 'STOI': s,
                    '균일축소 ECAPA': 0.479 + 3.36 * (s - 0.892),
                    '균일축소 Resem': 0.840 + 1.27 * (s - 0.892)})
    print(f"[{tag}] 완료")

print(pd.DataFrame(summary).round(3).to_string(index=False))

## 그림: 마스킹 vs 균일 축소 (입력 단계)

아래 숫자는 위 실험들의 평균값을 옮겨 적은 것이다. 오른쪽 아래로 갈수록 좋다(명료하고 보호가 강함).

In [ ]:
import matplotlib.pyplot as plt

uni  = [(0.680,-0.012,0.605),(0.735,0.049,0.653),(0.803,0.180,0.727),(0.892,0.479,0.840)]
post = [(0.967,0.958),(0.950,0.931),(0.928,0.893),(0.900,0.856),
        (0.872,0.817),(0.848,0.768),(0.827,0.694)]
trn  = {'trained k=16':(0.904,0.747,0.873), 'trained k=16 no-perc':(0.904,0.746,0.877),
        'trained k=64':(0.869,0.579,0.828)}

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for i, (col, name) in enumerate([(1, 'ECAPA similarity'), (2, 'Resemblyzer similarity')]):
    ax[i].plot([u[0] for u in uni], [u[col] for u in uni], 'o-', label='uniform scaling (control)')
    if col == 1:
        ax[i].plot([p[0] for p in post], [p[1] for p in post], 's--', label='post-hoc masking')
    for k, v in trn.items():
        ax[i].plot(v[0], v[col], '^', ms=9, label=k)
    ax[i].set_xlabel('STOI (higher = more intelligible)')
    ax[i].set_ylabel(f'{name} (lower = stronger protection)')
    ax[i].grid(alpha=.3)
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(ROOT / 'mask_exp' / 'masking_vs_uniform.png', dpi=200)
plt.show()

## 다음: 정화 → 복제

이 폴더의 `02_purify_v1.ipynb`(입력 `mask_exp/train_k64/D2`)와 `03_clone_v1.ipynb`(`RUN_NAME = enkidu_mask_k64`, C1·C4 재사용)를 순서대로 실행한다. 두 노트북은 `notebooks/`의 `02`, `03`에서 경로와 재사용 칸만 바꾼 것이다.